# Engenharia de Features (v3)
## ELO + Peso por Prestígio do Torneio

**Estratégia v3:** Combinar dois pesos simultaneamente em uma única feature:

```
gol_ponderado = gols × peso_torneio × (elo_adversario / 1000)
```

Onde `peso_torneio` é calculado pelo **ELO médio histórico** dos participantes de cada torneio, normalizado entre 0.5 e 1.0. Torneios com seleções mais fortes recebem peso maior — sem subjetividade, tudo baseado nos dados.

**Por que essa abordagem é melhor que a v2?**
- v2 adicionou 6 features novas → overfitting com dataset pequeno
- v3 adiciona apenas 2 features novas → menos risco, mais sinal
- A ponderação dupla (torneio + adversário) é mais rica semanticamente

**Dataset de saída:** `data/processed/features_completo_v3.csv`

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src/features')
from elo import calcular_elo_historico, calcular_peso_torneio

sns.set_theme(style='whitegrid')

df_raw = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
print(f'Shape: {df_raw.shape}')

## 2. Calculando ELO e Pesos dos Torneios

In [ ]:
# Calcular ELO histórico
df = calcular_elo_historico(df_raw, elo_inicial=1000, k_competitivo=40, k_amistoso=20)

# Calcular pesos dos torneios
pesos_torneio = calcular_peso_torneio(df, peso_min=0.5, peso_max=1.0)

# Salvar pesos para referência
pesos_df = pesos_torneio.reset_index()
pesos_df.columns = ['tournament', 'peso']
pesos_df = pesos_df.sort_values('peso', ascending=False)
pesos_df.to_csv('../data/processed/pesos_torneios.csv', index=False)

print('Top 20 torneios por peso de prestígio:')
print(pesos_df.head(20).to_string(index=False))

## 3. Visualização — Distribuição dos Pesos por Torneio

In [ ]:
# Top 25 torneios mais relevantes
top25 = pesos_df.head(25)

plt.figure(figsize=(12, 8))
bars = plt.barh(top25['tournament'][::-1], top25['peso'][::-1],
                color='steelblue', edgecolor='black', alpha=0.85)

for bar, val in zip(bars, top25['peso'][::-1]):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=8)

plt.xlabel('Peso de Prestígio (baseado no ELO médio dos participantes)')
plt.title('Peso por Torneio — Top 25\n(Calculado pelo ELO médio histórico dos participantes)', fontsize=12)
plt.xlim(0, 1.15)
plt.tight_layout()
plt.savefig('../article/figures/pesos_torneios.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Função de Features v3

A feature principal da v3 é:
```
gol_ponderado = gols × peso_torneio × (elo_adversario / 1000)
```

Isso captura simultaneamente:
- **Qualidade do adversário** (ELO)
- **Importância do torneio** (peso)

Um gol na Copa América contra a Argentina vale muito mais que um gol em amistoso contra um time fraco.

In [ ]:
def extrair_stats_v3(jogos, selecao, pesos_torneio):
    """Extrai stats com ponderação dupla: ELO do adversário × peso do torneio."""
    gm, gs, vit, elo_adv, pesos = [], [], [], [], []

    for _, row in jogos.iterrows():
        peso = pesos_torneio.get(row['tournament'], 0.5)

        if row['home_team'] == selecao:
            gm.append(row['home_score'])
            gs.append(row['away_score'])
            vit.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv.append(row['elo_away_antes'])
        else:
            gm.append(row['away_score'])
            gs.append(row['home_score'])
            vit.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv.append(row['elo_home_antes'])

        pesos.append(peso)

    gm     = np.array(gm)
    gs     = np.array(gs)
    vit    = np.array(vit)
    elo_adv = np.array(elo_adv)
    pesos  = np.array(pesos)

    return gm, gs, vit, elo_adv, pesos


def calcular_features_v3(selecao, ciclo, copa, pesos_torneio):
    """Calcula features v3: features v1 + ponderação dupla (ELO × torneio)."""

    jogos = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    gm, gs, vit, elo_adv, pesos = extrair_stats_v3(jogos, selecao, pesos_torneio)

    ultimos15 = jogos.tail(15)
    gm15, gs15, vit15, elo_adv15, pesos15 = extrair_stats_v3(ultimos15, selecao, pesos_torneio)

    # Ponderação dupla: gols × peso_torneio × (elo_adversario / 1000)
    gols_pond_ciclo  = (gm  * pesos  * (elo_adv  / 1000)).mean()
    gols_pond_ult15  = (gm15 * pesos15 * (elo_adv15 / 1000)).mean()

    # Target
    selecao_copa = copa[
        (copa['home_team'] == selecao) |
        (copa['away_team'] == selecao)
    ]
    gols_copa = [
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in selecao_copa.iterrows()
    ]

    return {
        # Features v1 (mantidas)
        'media_gols_marcados_ciclo':   gm.mean(),
        'media_gols_sofridos_ciclo':   gs.mean(),
        'pct_vitorias_ciclo':          vit.mean(),
        'total_jogos_ciclo':           len(jogos),
        'media_gols_marcados_ult15':   gm15.mean(),
        'media_gols_sofridos_ult15':   gs15.mean(),
        'pct_vitorias_ult15':          vit15.mean(),
        # Features v3 novas
        'gols_pond_torneio_elo_ciclo': gols_pond_ciclo,
        'gols_pond_torneio_elo_ult15': gols_pond_ult15,
        # Target
        'media_gols_copa':             np.mean(gols_copa)
    }

print('Funções v3 definidas!')

### Teste com o Brasil — Copa 2022

In [ ]:
pesos_dict = pesos_torneio.to_dict()

copa_2022 = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'].dt.year == 2022)
]
ciclo_2022 = df[
    (df['date'] >= '2018-07-16') &
    (df['date'] <= '2022-11-19') &
    (df['tournament'] != 'FIFA World Cup')
]

resultado_brasil = calcular_features_v3('Brazil', ciclo_2022, copa_2022, pesos_dict)
print('Features v3 — Brasil (Copa 2022):')
for k, v in resultado_brasil.items():
    print(f'  {k:<35} {v:.4f}')

## 5. Pipeline Completo — Todas as Copas (1994–2022)

In [ ]:
copas = {
    1994: {'ciclo_inicio': '1990-07-09', 'ciclo_fim': '1994-06-16', 'copa_inicio': '1994-06-17', 'copa_fim': '1994-07-17'},
    1998: {'ciclo_inicio': '1994-07-18', 'ciclo_fim': '1998-06-09', 'copa_inicio': '1998-06-10', 'copa_fim': '1998-07-12'},
    2002: {'ciclo_inicio': '1998-07-13', 'ciclo_fim': '2002-05-30', 'copa_inicio': '2002-05-31', 'copa_fim': '2002-06-30'},
    2006: {'ciclo_inicio': '2002-07-01', 'ciclo_fim': '2006-06-08', 'copa_inicio': '2006-06-09', 'copa_fim': '2006-07-09'},
    2010: {'ciclo_inicio': '2006-07-10', 'ciclo_fim': '2010-06-10', 'copa_inicio': '2010-06-11', 'copa_fim': '2010-07-11'},
    2014: {'ciclo_inicio': '2010-07-12', 'ciclo_fim': '2014-06-11', 'copa_inicio': '2014-06-12', 'copa_fim': '2014-07-13'},
    2018: {'ciclo_inicio': '2014-07-14', 'ciclo_fim': '2018-06-13', 'copa_inicio': '2018-06-14', 'copa_fim': '2018-07-15'},
    2022: {'ciclo_inicio': '2018-07-16', 'ciclo_fim': '2022-11-19', 'copa_inicio': '2022-11-20', 'copa_fim': '2022-12-18'},
}

todos_dados = []

for ano, datas in copas.items():
    print(f'Processando Copa {ano}...')

    ciclo = df[
        (df['date'] >= datas['ciclo_inicio']) &
        (df['date'] <= datas['ciclo_fim']) &
        (df['tournament'] != 'FIFA World Cup')
    ]
    copa = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= datas['copa_inicio']) &
        (df['date'] <= datas['copa_fim'])
    ]

    selecoes = pd.unique(copa[['home_team', 'away_team']].values.ravel())

    for selecao in selecoes:
        try:
            resultado = calcular_features_v3(selecao, ciclo, copa, pesos_dict)
            resultado['selecao']   = selecao
            resultado['copa_alvo'] = ano
            todos_dados.append(resultado)
        except Exception as e:
            print(f'  Erro em {selecao}: {e}')

df_final = pd.DataFrame(todos_dados)
print(f'\nDataset final: {df_final.shape[0]} linhas × {df_final.shape[1]} colunas')
df_final.head()

## 6. Validação e Salvamento

In [ ]:
print('Linhas por Copa:')
print(df_final['copa_alvo'].value_counts().sort_index())

print('\nNovas features v3 — Brasil 2022:')
brasil = df_final[
    (df_final['selecao'] == 'Brazil') &
    (df_final['copa_alvo'] == 2022)
][['media_gols_marcados_ciclo', 'gols_pond_torneio_elo_ciclo', 'gols_pond_torneio_elo_ult15']]
print(brasil.to_string(index=False))

df_final.to_csv('../data/processed/features_completo_v3.csv', index=False)
print('\nDataset v3 salvo em: data/processed/features_completo_v3.csv')

## 7. Resumo das Features

| Feature | Versão | Descrição |
|---------|--------|-----------|
| `media_gols_marcados_ciclo` | v1 | Média de gols marcados no ciclo completo |
| `media_gols_sofridos_ciclo` | v1 | Média de gols sofridos no ciclo completo |
| `pct_vitorias_ciclo` | v1 | % de vitórias no ciclo |
| `total_jogos_ciclo` | v1 | Total de jogos no ciclo |
| `media_gols_marcados_ult15` | v1 | Média de gols nos últimos 15 jogos |
| `media_gols_sofridos_ult15` | v1 | Média de gols sofridos nos últimos 15 |
| `pct_vitorias_ult15` | v1 | % de vitórias nos últimos 15 jogos |
| `gols_pond_torneio_elo_ciclo` | **v3** | gols × peso_torneio × (elo_adv/1000) — ciclo |
| `gols_pond_torneio_elo_ult15` | **v3** | gols × peso_torneio × (elo_adv/1000) — últ. 15 |

**Próximo passo:** `03_modelos_v3.ipynb` — comparar v1, v2 e v3.